# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`

This notebook demonstrates exploring the [FAIRˆ² dataset](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json) using the `mlcroissant` library. We will load the Croissant schema, examine available record sets and fields by their `@id`, extract the data into DataFrames, apply exploratory data analysis, and visualize key relationships.

### Dataset Source
The dataset source is provided via the following Croissant schema URL:

```
https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json
```


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install --quiet mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# Access the metadata object (not as dict)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}\n")
print(f"Version: {metadata.version}")

## 2. Data Overview

Review available record sets, fields (columns), and their IDs.

All references to fields and record sets use their `@id` for clarity and consistency.

In [ ]:
# List all available record sets and their columns by @id

record_sets_info = []
for record_set in dataset.record_sets:
    rec_id = record_set.id
    rec_name = record_set.name if hasattr(record_set, 'name') else ''
    columns = [col.id for col in record_set.columns]
    record_sets_info.append({'@id': rec_id, 'name': rec_name, 'columns': columns})

if not record_sets_info:
    print("No record sets found in the dataset. If the dataset provides data through other means, check the documentation or metadata.")
else:
    print("Available record sets and their columns (@id):\n")
    for info in record_sets_info:
        print(f"Record Set @id: {info['@id']}")
        print(f"  Columns (@id): {info['columns']}")


## 3. Data Extraction

Load data from each record set into a DataFrame for analysis. The record set and field `@id`s printed above can be used for reference.

If there are no record sets, skip to the metadata overview.

In [ ]:
# Extract data from each record set (@id)

dataframes = {}
record_set_ids = [rec['@id'] for rec in record_sets_info] if record_sets_info else []

for rec_id in record_set_ids:
    try:
        # Load all records for this record set
        records = list(dataset.records(record_set=rec_id))
        df = pd.DataFrame(records)
        dataframes[rec_id] = df
        print(f"Loaded {len(df)} rows for record set {rec_id}")
        print(f"  Columns: {df.columns.tolist()}\n")
    except Exception as e:
        print(f"Could not load records for {rec_id}: {e}")

# Display head of the first record set, if available
if dataframes:
    first_id = record_set_ids[0]
    print(f"Sample records from record set {first_id}:")
    display(dataframes[first_id].head())
else:
    print("No dataframes loaded. Please check for available record sets in the previous cell.")

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data. All references use their `@id`.

In [ ]:
# Example EDA: Filter, normalize, group by (if numeric fields exist)

if dataframes:
    record_set_id = record_set_ids[0]  # Use first available record set
    df = dataframes[record_set_id]
    # Find a numeric field (column) by checking dtypes
    numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if numeric_fields:
        numeric_field_id = numeric_fields[0]
        # Set arbitrary threshold (e.g., 10)
        threshold = 10
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold} (using @id):\n{filtered_df.head()}\n")

        # Normalize the numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
            filtered_df[numeric_field_id].std()
        )
        print(f"Normalized {numeric_field_id} (using @id):\n{filtered_df[[numeric_field_id, f'{numeric_field_id}_normalized']].head()}\n")

        # Attempt to group by a non-numeric field
        group_field_candidates = [col for col in df.columns if col != numeric_field_id and not pd.api.types.is_numeric_dtype(df[col])]
        if group_field_candidates:
            group_field_id = group_field_candidates[0]
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped mean of {numeric_field_id} by {group_field_id} (@id):\n{grouped_df.head()}")
        else:
            print("No suitable non-numeric columns found for grouping.")
    else:
        print("No numeric fields found in the first record set.")
else:
    print("No dataframes available for EDA.")

## 5. Visualization

Visualize data distributions or relationships between fields in the dataset. 

We use default matplotlib for plotting, if fields permit.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes:
    record_set_id = record_set_ids[0]
    df = dataframes[record_set_id]
    numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if numeric_fields:
        plt.figure(figsize=(8, 5))
        sns.histplot(df[numeric_fields[0]], kde=True)
        plt.title(f"Distribution of {numeric_fields[0]} (@id)")
        plt.xlabel(numeric_fields[0])
        plt.show()
        if len(numeric_fields) > 1:
            plt.figure(figsize=(8, 6))
            sns.scatterplot(x=df[numeric_fields[0]], y=df[numeric_fields[1]])
            plt.xlabel(numeric_fields[0])
            plt.ylabel(numeric_fields[1])
            plt.title(f"Scatterplot: {numeric_fields[0]} vs. {numeric_fields[1]} (@id)")
            plt.show()
    else:
        print("No numeric fields available for visualization.")
else:
    print("No data available for visualization.")

## 6. Conclusion

In this notebook, we:
- Loaded dataset metadata and examined its description and version from the Croissant schema.
- Explored available record sets and fields, referenced consistently by their `@id`.
- Extracted tabular data for analysis.
- Applied simple data transformations: filtering, normalization, and grouping.
- Visualized data distributions and (optionally) variable relationships.

Further exploration might include detailed statistical analysis, feature engineering, or advanced modeling as guided by research questions and dataset documentation.